# Measurement instrument for code reviews

This notebook demonstrates a measurement instrument which is organized into three parts:
* extraction from the source system
* saving into a raw data file
* calculation of the base measure

This measurement instrument extracts review information from gerrit database as a json string, saves it in a .csv file and then it calculates the number of reviews. 

The main difference for this measurement instrument is that it accesses the database directly, and needs to find the information about the measured entity. 

Here, the important part is to define the measured entity and the measured attribute, as we extract much more information from the database than we actually need. This particular measurement instrument exports and calculates the data for ALL entities of a specific type, not only the selected one (as in Example 2). 

* Measured entity: gromacs review 'gromacs~master~I0181ff67065c75f20cddc361f695df9bf888cd88'
* Measured attribute: review size
* Base measure: number_of_reviewed_lines

## Part 1: Configuration

In [56]:
# Gerrit and JSON specific configuration
from requests.auth import HTTPDigestAuth
from pygerrit2 import GerritRestAPI, HTTPBasicAuth
from IPython.display import clear_output
import requests
import pprint
import urllib

# name of the gerrit database
gerrit_url = "https://gerrit.gromacs.org"

# name of the result file
fileName = "./raw_data_gromacs.csv"

# pretty printer for json files
# used only when printing changes that have some problems
pp = pprint.PrettyPrinter(indent=2)

#auth = HTTPBasicAuth('username', 'password')
auth = None

# empty the file where we store the result
fileHandle = open(fileName, 'w', encoding = 'utf-8')

In [57]:
number_of_reviewed_lines = 0
measured_entity = 'gromacs~master~I0181ff67065c75f20cddc361f695df9bf888cd88'

### Exporting the data per review 

In [58]:
# header row
fileHandle.write('change_id;revision-id;filename;line;start_line;end_line;LOC;message\n')

# connecting to gerrit
rest = GerritRestAPI(url=gerrit_url, auth = auth)

In [59]:
changes = rest.get("/changes/?q=status:merged&o=ALL_FILES&o=ALL_REVISIONS&o=DETAILED_LABELS&start=0", 
                       headers={'Content-Type': 'application/json'})

In [60]:
# retrieving the length of the JSON file 
number_of_changes = len(changes)

# here we process the changes
for iIndex, change in enumerate(changes, start=1):
    changeID = change['id']

    if iIndex % 100 == 0:
        print("INFO: Extracting change: " + str(iIndex) + " of " + str(number_of_changes) )

    revisions = change['revisions']

    for revID in list(revisions.keys()):
        currentComment = rest.get("/changes/{}/revisions/{}/comments".format(changeID,revID), headers={'Content-Type': 'application/json'})

        # not all revisions have comments, so we only look for those that have them
        if len(currentComment) > 0:                
            for oneFile, oneComment in currentComment.items():                    
                try:
                    # this code extracts information about the comment 
                    # things like which file and which lines
                    for oneCommentItem in oneComment:
                        strFile = oneFile

                        # a few if-s because not always all parameters are there
                        if 'line' in oneCommentItem:
                            strLine = oneCommentItem['line']
                        else:
                            strLine = ''

                        if 'message' in oneCommentItem:
                            strMessage = oneCommentItem['message']
                        else:
                            strMessage = ''

                        # if there is a specific line and characters as comments
                        if 'range' in oneCommentItem:
                            strStartLine = oneCommentItem['range']['start_line']
                            strStartChar = oneCommentItem['range']['start_character']
                            strEndLine = oneCommentItem['range']['end_line']
                            strEndChar = oneCommentItem['range']['end_character']                        
                        else:                            
                            strStartLine = '0'
                            strStartChar = '0'
                            strEndLine = '0'
                            strEndChar = '0'

                        # if we can extract something from a file
                        # then here is where we do it
                        if strLine != '':
                            # we need the line below to properly encode the filename as URL
                            urlFileID = urllib.parse.quote_plus(strFile)
                            fileContentString = f'/changes/{changeID}/revisions/{revID}/files/{urlFileID}/content'
                            fileContent = rest.get(fileContentString, headers={'Content-Type': 'application/json'})
                            fileLines = fileContent.split("\n")

                            # if we have the lines delimitations (comment that is linked to lines)
                            if strStartLine != '0':
                                iStartLine = int(strStartLine) - 1
                                if strEndLine != '0':
                                    iEndLine = int(strEndLine) - 1  
                                else: 
                                    iEndLine = len(fileLines) - 1

                                for oneLine in fileLines[iStartLine:iEndLine]:
                                    strToCSV = str(changeID) + ";" + \
                                       str(revID) + ";" + \
                                       strFile + ";" + \
                                       str(strLine) + ";" + \
                                       str(strStartLine) + ";" + \
                                       str(strEndLine) + ";" + \
                                       oneLine.replace("\n", " _ ").replace('\r', '_').replace(';', '_') + ";" + \
                                       strMessage.replace("\n", " _ ").replace('\r', '_').replace(';', '_')
                                    fileHandle.write(strToCSV + "\n")
                            elif int(strLine) < len(fileLines):                                
                                # and if there are no delimitation, but there is a starting line
                                # and the starting line is below the end of the file
                                oneLine = fileLines[int(strLine)-1]
                                strToCSV = str(changeID) + ";" + \
                                           str(revID) + ";" + \
                                           strFile + ";" + \
                                           str(strLine) + ";" + \
                                           str(strStartLine) + ";" + \
                                           str(strEndLine) + ";" + \
                                           oneLine.replace("\n", " _ ").replace('\r', '_').replace(';', '_') + ";" + \
                                           strMessage.replace("\n", " _ ").replace('\r', '_').replace(';', '_')
                                fileHandle.write(strToCSV + "\n")
                        else: 
                            # there is no line specified, then we take the comment for the entire file
                            for oneLine in fileLines:
                                    strToCSV = str(changeID) + ";" + \
                                       str(revID) + ";" + \
                                       strFile + ";" + \
                                       str(strLine) + ";" + \
                                       str(strStartLine) + ";" + \
                                       str(strEndLine) + ";" + \
                                       oneLine.replace("\n", " _ ").replace('\r', '_').replace(';', '_') + ";" + \
                                       strMessage.replace("\n", " _ ").replace('\r', '_').replace(';', '_')
                                    fileHandle.write(strToCSV + "\n")

                except:
                    # this is a brutal exception handling, but we cannot check for all problems
                    print('INFO: Unhandled exception, moving on')
                    pp.pprint(oneComment)

INFO: Extracting change: 100 of 500
INFO: Extracting change: 200 of 500
INFO: Extracting change: 300 of 500
INFO: Extracting change: 400 of 500
INFO: Extracting change: 500 of 500


In this piece of code, we have created a new .csv file which contains a _relevant_ dump of the database of gromacs reviews. 

Let's read this file into a pandas data frame and see how it looks like. 

In [61]:
import pandas as pd

dfReviewRawData = pd.read_csv(fileName, sep=';')

In [62]:
dfReviewRawData.head(3)

,change_id,revision-id,filename,line,start_line,end_line,LOC,message
0,gromacs~master~Iac0b13d7db15f04a8b0b464df9fa13...,69f350744e40d375283234a2afbe8a238aa53bee,src/gromacs/fileio/xvgr.cpp,826.0,822,826,NaN,don't think this is needed because you can alw...
1,gromacs~master~Iac0b13d7db15f04a8b0b464df9fa13...,69f350744e40d375283234a2afbe8a238aa53bee,src/gromacs/fileio/xvgr.cpp,826.0,822,826,"gmx::MultiDimArray<std::vector<double>, gmx::d...",don't think this is needed because you can alw...
2,gromacs~master~Iac0b13d7db15f04a8b0b464df9fa13...,69f350744e40d375283234a2afbe8a238aa53bee,src/gromacs/fileio/xvgr.cpp,826.0,822,826,{,don't think this is needed because you can alw...


## Part 2: measurement instrument code

Now, the trick is that we would like to see how many comments we have per each patch. A patch is the same as change_id.

So, we need to make a table where we can get this data. This is the first part of this measurement instrument.

`dfCommentsPerPath` is the variable that contains the table with the values for all patches.  

In [63]:
dfRawDataUnique = dfReviewRawData.groupby(['change_id'])
dfCommentsPerPatch = dfRawDataUnique['message'].count()

We can also add other measures to the file with raw data, e.g. number of commented lines. 

In [64]:
dfLinesPerPatch = dfRawDataUnique['LOC'].count()

dfResultingDataSet = pd.concat([dfCommentsPerPatch, dfLinesPerPatch], axis=1)

dfResultingDataSet.head()

,message,LOC
change_id,,
gromacs~master~I0181ff67065c75f20cddc361f695df9bf888cd88,207,198
gromacs~master~I02b15beddebc160f2fe4fc21da64975977855699,2,0
gromacs~master~I05f577e76ae8cb4703cbf3b17a101716be4593de,2,2
gromacs~master~I0c0129fd881c84d366c1745ee3d9e3a8caf8633e,17,17
gromacs~master~I1687981cc80e2388714cbbb3113f37e34582e31c,3,3


## Part 3: assigning the value to the base measure

In [65]:
number_of_reviewed_lines = dfResultingDataSet.loc[measured_entity]['LOC']

## Part 4: Further processing

In [66]:
print(f'Number of commented lines for entity {measured_entity} is {number_of_reviewed_lines}')

Number of commented lines for entity gromacs~master~I0181ff67065c75f20cddc361f695df9bf888cd88 is 198


## Considerations

This demo showed how to extract data about a specific project and make a data set of it. The measured entity, however, was one particular review. 

I would like to discuss a few things here:
* What would happen if the measured entity was the entire gromacs project?
* Can we use the same measures then?
    * If yes, what does it mean for the definition of these measures?
    * If not, which other measures can we use?
